# Lesson 2: Data Cleaning & Loading


---
## 1. Setup

In [ ]:
import sqlite3
import math
import pandas as pd
from datetime import datetime, time, timedelta
from opensearchpy import OpenSearch, helpers
from opensearch_dsl import Document, Keyword, Date, Float, Boolean, Search

pd.set_option('display.width', 300)
pd.set_option('display.max_columns', 20)

In [ ]:
con = sqlite3.connect('database.db')

machine_log    = pd.read_sql_query('SELECT * FROM machine_log', con)
maintenance    = pd.read_sql_query('SELECT * FROM maintenance', con)
shift_schedule = pd.read_sql_query('SELECT * FROM shift_schedule', con)

con.close()

print(f'machine_log:    {machine_log.shape}')
print(f'maintenance:    {maintenance.shape}')
print(f'shift_schedule: {shift_schedule.shape}')

---
## 2. Domain: `machine_log`

**Goal:** Convert the two timestamp strings into proper `datetime` objects.

This is the simplest domain — no missing values, no format ambiguity. It's a pure type conversion.

### Dirty Input

In [ ]:
machine_log.head(5)

In [ ]:
print('dtypes before cleaning:')
print(machine_log.dtypes)
print(f'\nMissing values: {machine_log.isna().sum().sum()}')

### Cleaning

`pd.to_datetime()` handles ISO 8601 strings like `2025-09-01T06:20:00` automatically. We convert both columns, drop the originals, and rename to snake_case.

In [ ]:
machine_log['time_in']  = pd.to_datetime(machine_log['Time_in'])
machine_log['time_out'] = pd.to_datetime(machine_log['Time_out'])

machine_log.drop(columns=['Time_in', 'Time_out'], inplace=True)

# Rename ID columns to snake_case for consistency
machine_log.rename(columns={'employee_ID': 'employee_id', 'machine_ID': 'machine_id'}, inplace=True)

### Clean Output ✓

In [ ]:
display(machine_log.head(5))
print()
machine_log.info()

---
## 3. Domain: `maintenance`

**Goal:** Parse timestamps, handle missing values, and correct records where the fix appears to precede the request.

This domain has three distinct data quality issues, each requiring a different strategy.

### Dirty Input

In [ ]:
maintenance.head(8)

In [ ]:
print('Missing values:')
print(maintenance.isna().sum())

print(f'\nRows with missing fix_timestamp:')
display(maintenance[maintenance['fix_timestamp'].isna()][['request_timestamp', 'fix_timestamp', 'requestor_ID']].head(5))

In [ ]:
# Illustrate the two timestamp formats side-by-side
with_seconds    = maintenance[maintenance['request_timestamp'].str.len() > 14]['request_timestamp'].head(3)
without_seconds = maintenance[maintenance['request_timestamp'].str.len() == 14]['request_timestamp'].head(3)

print('With seconds    :', list(with_seconds))
print('Without seconds :', list(without_seconds))

### Cleaning — Step 1: Parse Inconsistent Timestamps

The format is `MM/DD/YY HH:MM:SS` when seconds are present, and `MM/DD/YY HH:MM` when they are not. `pd.to_datetime` with a single format string would fail on one of them, so we inspect the length to pick the right format.

In [ ]:
def parse_maint_ts(ts_str):
    if pd.isna(ts_str):
        return pd.NaT
    ts_str = ts_str.strip()
    if len(ts_str) == 14:  # 'MM/DD/YY HH:MM' — no seconds
        return pd.to_datetime(ts_str, format='%m/%d/%y %H:%M')
    return pd.to_datetime(ts_str, format='%m/%d/%y %H:%M:%S')

maintenance['request_timestamp'] = maintenance['request_timestamp'].apply(parse_maint_ts) # type: ignore
maintenance['fix_timestamp']     = maintenance['fix_timestamp'].apply(parse_maint_ts) # type: ignore

print('Dtypes after parsing:')
print(maintenance[['request_timestamp', 'fix_timestamp']].dtypes)

In [ ]:
# Spot-check a few rows with/without seconds
maintenance[['request_timestamp', 'fix_timestamp']].head(5)

### Cleaning — Step 2: Impute Missing `fix_timestamp`

13 rows have no `fix_timestamp` — the repair was logged but the completion time was not recorded. 

**Design decision:** Use `request_timestamp + 4 hours` as a reasonable proxy for typical repair turnaround time. We document this assumption so anyone reading the data knows these are imputed, not observed.

In [ ]:
# Capture the null indexes so we can verify the fix
null_fix_idx = maintenance[maintenance['fix_timestamp'].isna()].index
print(f'Rows with missing fix_timestamp: {len(null_fix_idx)}')

# Impute: request_timestamp + 4 hours
maintenance.loc[null_fix_idx, 'fix_timestamp'] = (
    maintenance.loc[null_fix_idx, 'request_timestamp'] + pd.to_timedelta('4 hours')
)

assert maintenance['fix_timestamp'].isna().sum() == 0, 'Still have nulls!'
print('No missing fix_timestamps remaining ✓')

In [ ]:
# Show the imputed rows — fix_timestamp is exactly 4 hours after request
maintenance.loc[null_fix_idx, ['request_timestamp', 'fix_timestamp']].head(5)

### Cleaning — Step 3: Correct Time-Travel Records

16 rows have `fix_timestamp < request_timestamp` — the machine was supposedly fixed *before* the repair was requested. This is physically impossible; it indicates a data entry error (likely a date transposition or a copy-paste mistake).

**Design decision:** Replace the erroneous `fix_timestamp` with `request_timestamp + 30 minutes`. This is a conservative lower bound — 30 minutes is a plausible minimum repair time.

In [ ]:
early_fix_mask = maintenance['fix_timestamp'] < maintenance['request_timestamp']
print(f'Time-travel records: {early_fix_mask.sum()}')

maintenance.loc[early_fix_mask, ['request_timestamp', 'fix_timestamp']]

In [ ]:
maintenance.loc[early_fix_mask, 'fix_timestamp'] = (
    maintenance.loc[early_fix_mask, 'request_timestamp'] + pd.to_timedelta('30 minutes')
)

remaining = (maintenance['fix_timestamp'] < maintenance['request_timestamp']).sum()
assert remaining == 0, 'Still have time-travel records!'
print('All time-travel records corrected ✓')

### Clean Output ✓

In [ ]:
# Show the corrected time-travel rows side-by-side
print('Corrected rows (fix is now 30 min after request):')
display(maintenance.loc[early_fix_mask, ['request_timestamp', 'fix_timestamp']].head(5))

print('\nOverall info:')
maintenance.info()

---
## 4. Domain: `shift_schedule`

**Goal:** Standardize the `Title` column and parse `clock_in` / `clock_out` into full datetime objects.

This is the most complex domain. The clock time parsing requires knowledge of *which shift* a row belongs to in order to resolve the AM/PM ambiguity when the indicator is missing.

### Dirty Input

In [ ]:
print('Title value counts (all mean the same two roles):')
print(shift_schedule['Title'].value_counts())

In [ ]:
print('clock_in value counts (note entries with and without AM/PM):')
print(shift_schedule['clock_in'].value_counts().head(12))

In [ ]:
# A sample that shows all three shifts and both time formats
shift_schedule[['employee_ID', 'shift', 'date', 'clock_in', 'clock_out', 'Title']].head(10)

### Cleaning — Step 1: Normalize `Title`

Strip whitespace, lowercase, then map to canonical role names. We rename `line` → `Operator` to use a more descriptive term than the original shorthand.

In [ ]:
TITLE_MAP = {'line': 'Operator', 'manager': 'Manager'}

shift_schedule['title'] = (
    shift_schedule['Title']
    .str.strip()
    .str.lower()
    .map(TITLE_MAP)
)

shift_schedule.drop(columns=['Title'], inplace=True)

print('After normalization:')
print(shift_schedule['title'].value_counts())

### Cleaning — Step 2: Parse Clock Times

The shift definitions give us the context we need to resolve ambiguous times:

| Shift | Start range | End range |
|-------|-------------|----------|
| 1 (Day)   | ~6 AM  | ~2 PM  |
| 2 (Swing) | ~2 PM  | ~10 PM |
| 3 (Night) | ~10 PM | ~6 AM  |

When the AM/PM indicator is missing (`6:15` instead of `6:15 AM`), we compare the bare hour against the shift's expected hour range to determine whether to add 12.

In [ ]:
# Canonical shift start/end hours — used to resolve ambiguous times
SHIFT_HOURS = {
    1: {'clock_in': 6,  'clock_out': 14},
    2: {'clock_in': 14, 'clock_out': 22},
    3: {'clock_in': 22, 'clock_out': 6},
}

def adjust_hour(hour, shift, col):
    """Add 12 to a bare hour if the shift says it should be in the PM/night range."""
    expected = SHIFT_HOURS[shift][col]
    if hour <= 12 and expected > 12:
        return hour + 12
    return hour

def parse_clock_time(row, col):
    clock_str = row[col]
    date_dt   = pd.to_datetime(row['date'], format='%m/%d/%Y').date()

    if ' ' in clock_str:                         # '6:15 AM' / '10:00 PM'
        t = datetime.strptime(clock_str, '%I:%M %p').time()
    else:                                         # '6:15' / '10:00' — no AM/PM
        h, m = map(int, clock_str.split(':'))
        h = adjust_hour(h, row['shift'], col)
        t = time(h, m)

    return datetime.combine(date_dt, t)

shift_schedule['clock_in_dt']  = shift_schedule.apply(lambda r: parse_clock_time(r, 'clock_in'),  axis=1)
shift_schedule['clock_out_dt'] = shift_schedule.apply(lambda r: parse_clock_time(r, 'clock_out'), axis=1)

shift_schedule.drop(columns=['clock_in', 'clock_out', 'date'], inplace=True)
shift_schedule.rename(columns={'employee_ID': 'employee_id'}, inplace=True)

### Clean Output ✓

In [ ]:
print('title counts:')
print(shift_schedule['title'].value_counts())
print()

# One row from each shift to verify times look right
sample = shift_schedule.groupby('shift').first().reset_index()
display(sample[['shift', 'employee_id', 'clock_in_dt', 'clock_out_dt', 'title']])
print()
shift_schedule.info()

---
## 5. Building the index

Now that each domain is clean we integrate them into a single unified table: **`machine_usage`**.

The key challenge is that the source tables don't share a direct foreign key. We use **`pd.merge_asof`** — a time-ordered nearest-match join — to correlate records across tables.

The full schema of the index (Step 6 adds precomputed analytics fields on top of this):

| Column | Source | Description |
|--------|--------|-------------|
| `id` | computed | Composite key: `{machine_id}_{start}_{end}` |
| `machine_id` | machine_log / maintenance→machine_log | Machine UUID |
| `personnel` | machine_log.employee_id / maintenance.requestor_id | Person UUID |
| `title` | shift_schedule.title / hardcoded | `Operator`, `Manager`, or `Maintenance` |
| `type` | hardcoded | `Regular` or `Maintenance` |
| `start_timestamp` | machine_log.time_in / maintenance.request_timestamp | |
| `end_timestamp` | machine_log.time_out / maintenance.fix_timestamp | |
| `shift_name` | derived from `start_timestamp` hour | `Day`, `Swing`, or `Night` |
| `reason_code` | maintenance.reason_code | Maintenance records only; `null` for Regular |

### Step 1: Add `title` to Machine Log Records

For each machine log entry, find the shift schedule record for the same employee whose `clock_in_dt` is the latest one **on or before** the machine log `time_in`. This gives us the title (Operator / Manager) in effect at that time.

In [ ]:
shift_sorted = shift_schedule[['employee_id', 'clock_in_dt', 'title']].sort_values('clock_in_dt')

machine_log_with_title = pd.merge_asof(
    machine_log.sort_values('time_in'),
    shift_sorted,
    left_on='time_in',
    right_on='clock_in_dt',
    by='employee_id',
    direction='backward',
)

print(f'Rows: {len(machine_log_with_title)}')
print(f'Title nulls before fix: {machine_log_with_title["title"].isna().sum()}')

# The backward merge only matches if a shift entry exists *before* time_in.
# The very first sessions of the day arrive before clock_in is logged, so no
# prior entry is found. Use each employee's title from shift_schedule as the
# fallback — this correctly preserves Manager vs Operator rather than
# blindly defaulting everything to Operator.
title_by_employee = shift_schedule.groupby('employee_id')['title'].first()

null_mask = machine_log_with_title['title'].isna()
machine_log_with_title.loc[null_mask, 'title'] = (
    machine_log_with_title.loc[null_mask, 'employee_id']
    .map(title_by_employee)
)

print(f'Title nulls after fix:  {machine_log_with_title["title"].isna().sum()}')
display(machine_log_with_title[['employee_id', 'machine_id', 'time_in', 'time_out', 'title']].head(5))

### Step 2: Add `machine_id` to Maintenance Records

Maintenance requests don't record which machine broke down. We infer it by finding the machine log entry where:
- The same person (`requestor_ID == employee_id`) was last seen
- Their session ended (`time_out`) **at or before** the request timestamp

The assumption: the machine you were just working on is the one you're reporting.

**Fallback:** Managers file reports but don't appear in `machine_log`. For those rows we fall back to the globally most recent machine log entry before the request.

In [ ]:
# Rename maintenance columns for the merge
maintenance.rename(columns={'requestor_ID': 'requestor_id'}, inplace=True)

machine_log_sorted = machine_log.sort_values('time_out')
maintenance_sorted = maintenance.sort_values('request_timestamp')

maintenance = pd.merge_asof(
    maintenance_sorted,
    machine_log_sorted[['employee_id', 'machine_id', 'time_out']],
    left_on='request_timestamp',
    right_on='time_out',
    left_by='requestor_id',
    right_by='employee_id',
    direction='backward',
)

unmatched = maintenance['machine_id'].isna().sum()
print(f'Matched on first pass: {len(maintenance) - unmatched}/{len(maintenance)}')
print(f'Unmatched (likely managers): {unmatched}')

In [ ]:
# Global fallback: most recent machine session before the request, regardless of who was on it
def find_global_match(request_dt):
    candidates = machine_log_sorted[machine_log_sorted['time_out'] <= request_dt]
    if candidates.empty:
        return pd.Series([None, None, None], index=['machine_id', 'employee_id', 'time_out'])
    return candidates.iloc[-1][['machine_id', 'employee_id', 'time_out']]

unmatched_mask = maintenance['machine_id'].isna()

maintenance.loc[unmatched_mask, ['machine_id', 'employee_id', 'time_out']] = (
    maintenance.loc[unmatched_mask, 'request_timestamp']
    .apply(find_global_match)
    .values
)

assert maintenance['machine_id'].isna().sum() == 0
print('All maintenance records have a machine_id ✓')

# Show a sample of the manager-submitted rows that got the fallback
display(maintenance.loc[unmatched_mask, ['requestor_id', 'request_timestamp', 'machine_id']].head(5))

### Step 3: Build the Unified `machine_usage` Table

Project each source into a common schema, then concatenate.

In [ ]:
def make_id(machine_id, start, end):
    return f'{machine_id}_{start.isoformat()}_{end.isoformat()}'

# Regular records
regular = machine_log_with_title[['machine_id', 'employee_id', 'title', 'time_in', 'time_out']].copy()
regular['id']   = regular.apply(lambda r: make_id(r.machine_id, r.time_in, r.time_out), axis=1)
regular['type'] = 'Regular'
regular.rename(columns={'employee_id': 'personnel', 'time_in': 'start_timestamp', 'time_out': 'end_timestamp'}, inplace=True)

# Maintenance records
maint = maintenance[['machine_id', 'requestor_id', 'request_timestamp', 'fix_timestamp']].copy()
maint['id']    = maint.apply(lambda r: make_id(r.machine_id, r.request_timestamp, r.fix_timestamp), axis=1)
maint['title'] = 'Maintenance'
maint['type']  = 'Maintenance'
maint.rename(columns={'requestor_id': 'personnel', 'request_timestamp': 'start_timestamp', 'fix_timestamp': 'end_timestamp'}, inplace=True)

machine_usage = pd.concat(
    [regular[['id', 'machine_id', 'personnel', 'title', 'type', 'start_timestamp', 'end_timestamp']],
     maint[  ['id', 'machine_id', 'personnel', 'title', 'type', 'start_timestamp', 'end_timestamp']]],
    ignore_index=True,
)

print(f'Total rows: {len(machine_usage)}')
print()
print('Type breakdown:')
print(machine_usage['type'].value_counts())
print()
print(f'Date range: {machine_usage["start_timestamp"].min()}  →  {machine_usage["end_timestamp"].max()}')

In [ ]:
machine_usage.head(8)

### Step 4: Add Derived Fields

Two fields can be added directly from data we already have — no aggregation or window logic needed:

- **`shift_name`** — derived from the hour of `start_timestamp`: `Day` (6–14), `Swing` (14–22), `Night` (otherwise)
- **`reason_code`** — for Maintenance records only; looked up by `(personnel, start_timestamp)` from the maintenance table

In [ ]:
# --- shift_name ---
def infer_shift_name(hour):
    if 6 <= hour < 14: return 'Day'
    if 14 <= hour < 22: return 'Swing'
    return 'Night'

machine_usage['shift_name'] = machine_usage['start_timestamp'].dt.hour.map(infer_shift_name)

# --- reason_code (Maintenance records only) ---
# maintenance still uses its original column name request_timestamp here;
# the rename to start_timestamp only exists on the maint slice inside step3-concat.
reason_lookup = dict(zip(
    zip(maintenance['requestor_id'], maintenance['request_timestamp']),
    maintenance['reason_code'],
))

machine_usage['reason_code'] = machine_usage.apply(
    lambda r: reason_lookup.get((r['personnel'], r['start_timestamp']))
              if r['type'] == 'Maintenance' else None,
    axis=1,
)

print('shift_name counts:', machine_usage['shift_name'].value_counts().to_dict())
print('reason_code non-null:', machine_usage['reason_code'].notna().sum(), '(should equal 198)')

# Show a mix of both types so reason_code population is visible
cols = ['machine_id', 'personnel', 'title', 'type', 'start_timestamp', 'shift_name', 'reason_code']
sample = pd.concat([
    machine_usage[machine_usage['type'] == 'Regular'].head(3),
    machine_usage[machine_usage['type'] == 'Maintenance'].head(3),
]).reset_index(drop=True)
display(sample[cols])

---
## 6. Load into OpenSearch

### Why precompute analytics fields at index time?

OpenSearch is a search engine, not a relational database. It does not support window functions, CTEs, or temporal JOINs. Fields we want to filter or aggregate on must be present in each document at index time.

This is the core tradeoff of the bronze layer: **compute at write time so reads are trivial.**

### Connect to OpenSearch

Make sure Docker is running: `docker compose -f opensearch/docker-compose.yml up -d`

In [ ]:
client = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    use_ssl=False,
)

_os_ready = False
try:
    info = client.info()
    print(f'Connected to OpenSearch {info["version"]["number"]} ✓')
    _os_ready = True
except Exception:
    print('❌ OpenSearch is not reachable on localhost:9200.')
    print('   Start it with: docker compose -f opensearch/docker-compose.yml up -d')
    print('   Then re-run this cell before continuing.')

### Define the Index Schema as a Document Class

Each field in `machine_usage` maps to a typed DSL field. The inner `Index` class holds the index name and settings — `MachineUsage.init(using=client)` reads these to create the index with the correct mapping.

In [ ]:
class MachineUsage(Document):
    machine_id                  = Keyword()
    personnel                   = Keyword()
    title                       = Keyword()
    type                        = Keyword()
    start_timestamp             = Date()
    end_timestamp               = Date()
    duration_hours              = Float()
    shift_name                  = Keyword()
    shift_active_hours          = Float()
    employee_idle_hours_from_prev = Float()
    machine_gap_hours_from_prev = Float()
    reason_code                 = Keyword()

    class Index:
        name = 'machine_usage'
        settings = {
            'number_of_shards': 1,
            'number_of_replicas': 0,
        }

print('MachineUsage document class defined ✓')

In [ ]:
assert _os_ready, 'OpenSearch is not connected — re-run the cell above first.'

# Delete first so re-running this notebook is idempotent
client.indices.delete(index=MachineUsage.Index.name)

# init() reads the Document class definition and creates the index with
# the correct mapping and settings — no hand-written dict required.
MachineUsage.init(using=client)
print(f'Index "{MachineUsage.Index.name}" created ✓')

### Enrichment: Precomputing Aggregate & Window Fields

The remaining fields require looking *across* multiple records — aggregation or LAG-style window logic that OpenSearch cannot perform at query time. We compute them here and store them in each document.

| Field | Description |
|-------|-------------|
| `duration_hours` | End minus start in fractional hours |
| `employee_idle_hours_from_prev` | Idle hours between the previous session's end and this session's start, for the same person within the same shift |
| `machine_gap_hours_from_prev` | Hours since the same machine's previous session (any type) |
| `shift_active_hours` | Total Regular hours for this person across all sessions in this shift |

In [ ]:
# --- duration_hours ---
machine_usage['duration_hours'] = (
    (machine_usage['end_timestamp'] - machine_usage['start_timestamp'])
    .dt.total_seconds() / 3600
).round(2)

print('duration_hours sample:', machine_usage['duration_hours'].head(3).tolist())

In [ ]:
# --- employee_idle_hours_from_prev ---
# For Regular records: idle time between the previous session's end and this session's start,
# for the same person within the same shift.
# Night shift sessions that start 00:00–05:59 belong to the *previous* day's canonical shift.

def canonical_shift_key(ts):
    h = ts.hour
    num = 1 if 6 <= h < 14 else (2 if 14 <= h < 22 else 3)
    d = ts.date()
    if num == 3 and h < 6:
        d -= timedelta(days=1)
    return (num, d)

machine_usage['_shift_key'] = machine_usage['start_timestamp'].apply(canonical_shift_key)

reg_mask = machine_usage['type'] == 'Regular'
reg = machine_usage[reg_mask].sort_values(['personnel', '_shift_key', 'start_timestamp'])
prev_end = reg.groupby(['personnel', '_shift_key'])['end_timestamp'].shift(1)
machine_usage.loc[reg.index, 'employee_idle_hours_from_prev'] = (
    (reg['start_timestamp'] - prev_end).dt.total_seconds() / 3600
).round(2)

# --- machine_gap_hours_from_prev ---
# For all records: gap from previous usage of the same machine.
all_s = machine_usage.sort_values(['machine_id', 'start_timestamp'])
prev_machine_end = all_s.groupby('machine_id')['end_timestamp'].shift(1)
machine_usage.loc[all_s.index, 'machine_gap_hours_from_prev'] = (
    (all_s['start_timestamp'] - prev_machine_end).dt.total_seconds() / 3600
).round(2)

gapped = machine_usage['employee_idle_hours_from_prev'].notna().sum()
print(f'Records with employee_idle_hours_from_prev: {gapped}')

In [ ]:
# --- shift_active_hours ---
# Total active Regular hours per (person, shift).

reg_df = machine_usage[reg_mask].copy()
reg_df['_dur'] = (reg_df['end_timestamp'] - reg_df['start_timestamp']).dt.total_seconds() / 3600

shift_totals = reg_df.groupby(['personnel', '_shift_key'])['_dur'].transform('sum').round(2)
machine_usage.loc[reg_mask, 'shift_active_hours'] = shift_totals.values

# Drop the helper column used by both gap cells
machine_usage.drop(columns=['_shift_key'], inplace=True)

print('Enrichment complete. Final columns:')
print(list(machine_usage.columns))

In [ ]:
cols = ['id', 'type', 'duration_hours', 'shift_name',
        'employee_idle_hours_from_prev', 'shift_active_hours',
        'machine_gap_hours_from_prev']

reg_with_gap = machine_usage[machine_usage['employee_idle_hours_from_prev'].notna()].head(3)
reg_first    = machine_usage[machine_usage['type'] == 'Regular'].iloc[4:6]
maint_rows   = machine_usage[machine_usage['type'] == 'Maintenance'].head(2)

sample = pd.concat([reg_first, reg_with_gap, maint_rows]).drop_duplicates().reset_index(drop=True)
display(sample[cols])

### Bulk Index

In [ ]:
def _null(val):
    """Convert pandas NaN to None — OpenSearch rejects float('nan') as invalid JSON."""
    try:
        return None if math.isnan(val) else val
    except TypeError:
        return val

def row_to_doc(row):
    return MachineUsage(
        meta={'id': row['id']},
        machine_id                    = row['machine_id'],
        personnel                     = row['personnel'],
        title                         = row['title'],
        type                          = row['type'],
        start_timestamp               = row['start_timestamp'].isoformat(),
        end_timestamp                 = row['end_timestamp'].isoformat(),
        duration_hours                = row['duration_hours'],
        shift_name                    = row['shift_name'],
        shift_active_hours            = _null(row.get('shift_active_hours')),
        employee_idle_hours_from_prev = _null(row.get('employee_idle_hours_from_prev')),
        machine_gap_hours_from_prev   = _null(row.get('machine_gap_hours_from_prev')),
        reason_code                   = _null(row.get('reason_code')),
    )

actions = (row_to_doc(row).to_dict(include_meta=True) for _, row in machine_usage.iterrows())
success, errors = helpers.bulk(client, actions)

client.indices.refresh(index=MachineUsage.Index.name)

if errors:
    print(f'WARNING: {len(errors)} errors during indexing')
else:
    doc_count = client.count(index=MachineUsage.Index.name)['count']
    print(f'Indexed {success} documents. Index count: {doc_count} ✓')

---
## 7. Exploring the index with the Search DSL

We query the index using `opensearch-dsl`'s `Search` class. The pattern:

```python
s = Search(using=client, index='machine_usage')
s = s.filter('term', field=value)   # narrow the result set
s = s.sort('field')                 # order results
s = s[:10]                          # size (slice notation)
resp = s.execute()

for hit in resp:                    # iterate hits directly
    print(hit.field_name)           # access fields as attributes
```

Aggregations are attached with `s.aggs.bucket(...)` and read from `resp.aggregations`.

### Query 1: Sample Documents (match_all)

The simplest possible query — returns any 5 documents. Good for a sanity check.

In [ ]:
s = Search(using=client, index=MachineUsage.Index.name)[:5]
resp = s.execute()

print(f'Total documents in index: {resp.hits.total.value}')
print()
for hit in resp:
    print(f"{hit.type:<15} {hit.shift_name:<7} {hit.start_timestamp[:19]}  {hit.duration_hours:.2f}h  {hit.machine_id[:8]}...")

### Query 2: Maintenance Records Only

`.filter('term', ...)` performs an exact-match on a `keyword` field. Chaining multiple `.filter()` calls applies them as `bool / filter` clauses — all must match, and results are not scored (so OpenSearch can cache the filter automatically).

In [ ]:
s = (
    Search(using=client, index=MachineUsage.Index.name)
    .filter('term', type='Maintenance')
    [:5]
)
resp = s.execute()

print(f'Maintenance records: {resp.hits.total.value}')
print()
for hit in resp:
    print(f"{hit.start_timestamp[:19]}  machine={hit.machine_id[:8]}...  duration={hit.duration_hours:.2f}h  reason={str(hit.reason_code)[:8]}...")

### Query 3: All Sessions for a Specific Machine

Filter by `machine_id` to see everything that has happened to one machine — both Regular operator sessions and Maintenance events, in time order.

In [ ]:
sample_machine = machine_usage['machine_id'].iloc[0]
print(f'Querying machine: {sample_machine}')

s = (
    Search(using=client, index=MachineUsage.Index.name)
    .filter('term', machine_id=sample_machine)
    .sort('start_timestamp')
    [:10]
)
resp = s.execute()

print(f'\nTotal sessions for this machine: {resp.hits.total.value}')
print()
print(f'{"Type":<15} {"Start":<22} {"Hours":>6}  Personnel')
print('-' * 70)
for hit in resp:
    print(f"{hit.type:<15} {hit.start_timestamp[:19]:<22} {hit.duration_hours:>6.2f}  {hit.personnel[:8]}...")

### Query 4: Top 3 Machines by Total Usage Hours

A `terms` aggregation groups documents by a `keyword` field (like `GROUP BY` in SQL). Nesting a `sum` metric aggregation inside it computes the total for each group.

In [ ]:
s = Search(using=client, index=MachineUsage.Index.name)[:0]
s.aggs.bucket(
    'by_machine', 'terms',
    field='machine_id', size=3,
    order={'total_hours': 'desc'},
).metric('total_hours', 'sum', field='duration_hours') # type: ignore

resp = s.execute()

print('Top 3 machines by total usage hours:\n')
print(f'{"Machine ID":<40} {"Total Hours":>12}  {"Sessions":>9}')
print('-' * 65)
for bucket in resp.aggregations.by_machine.buckets:
    print(f"{bucket.key:<40} {bucket.total_hours.value:>12.1f}  {bucket.doc_count:>9}")

### Query 5: Maintenance Events in October 2025

Chain two `.filter()` calls — one for the exact type match, one for the date range. The DSL compiles these into a `bool / filter` clause automatically, so no manual nesting is needed.

In [ ]:
s = (
    Search(using=client, index=MachineUsage.Index.name)
    .filter('term', type='Maintenance')
    .filter('range', start_timestamp={'gte': '2025-10-01', 'lt': '2025-11-01'})
    .sort('start_timestamp')
    [:20]
)
resp = s.execute()

print(f'Maintenance events in October 2025: {resp.hits.total.value}')
print()
print(f'{"Date":<22} {"Duration":>9}  {"Machine":<12}  Personnel')
print('-' * 70)
for hit in resp:
    print(f"{hit.start_timestamp[:19]:<22} {hit.duration_hours:>9.2f}h  {hit.machine_id[:8]+'...':<12}  {hit.personnel[:8]}...")

---
## Summary

In this lesson we went from three messy raw tables to a fully indexed, queryable index:

| Step | What we did |
|------|-------------|
| `machine_log` cleaning | Converted string timestamps → `datetime64` |
| `maintenance` cleaning | Parsed inconsistent formats, imputed 13 nulls, corrected 16 time-travel records |
| `shift_schedule` cleaning | Normalized mixed-case titles, resolved AM/PM ambiguity using shift context |
| Integration | Two `merge_asof` joins + global fallback → unified 814-row `machine_usage` table |
| Enrichment | Precomputed `duration_hours`, `shift_name`, gap fields, shift aggregates, `reason_code` |
| Load | Bulk indexed with explicit mapping → verified 814 documents |
| Queries | Demonstrated term, range, and aggregation DSL patterns |